In [0]:
from pyspark.sql.functions import *

# -----------------------------
# LOAD BRONZE TABLES
# -----------------------------

customers = spark.table("bronze.bronze_customers")
products = spark.table("bronze.bronze_products")
orders = spark.table("bronze.bronze_orders")
order_items = spark.table("bronze.bronze_order_items")
payments = spark.table("bronze.bronze_payments")
shipping = spark.table("bronze.bronze_shipping")

# -----------------------------
# CUSTOMERS (already learned pattern)
# -----------------------------

silver_customers = customers \
    .dropDuplicates(["customer_id"]) \
    .withColumn("customer_name", initcap(col("customer_name"))) \
    .withColumn("email", lower(col("email"))) \
    .withColumn("city", initcap(col("city"))) \
    .withColumn("country", initcap(col("country"))) \
    .withColumn("signup_date", to_date(col("signup_date"))) \
    .withColumn("signup_year", year(col("signup_date"))) \
    .filter(col("customer_id").isNotNull())

# -----------------------------
# PRODUCTS
# -----------------------------

silver_products = products \
    .dropDuplicates(["product_id"]) \
    .withColumn("product_name", initcap(col("product_name"))) \
    .withColumn("category", initcap(col("category"))) \
    .filter(col("price") > 0)

# -----------------------------
# ORDERS
# -----------------------------

silver_orders = orders \
    .dropDuplicates(["order_id"]) \
    .withColumn("order_date", to_date(col("order_date"))) \
    .withColumn("order_year", year(col("order_date"))) \
    .filter(col("customer_id").isNotNull())

# -----------------------------
# ORDER ITEMS
# -----------------------------

silver_order_items = order_items \
    .dropDuplicates(["order_item_id"]) \
    .filter(col("quantity") > 0)

# -----------------------------
# PAYMENTS
# -----------------------------

silver_payments = payments \
    .dropDuplicates(["payment_id"]) \
    .withColumn("payment_status", initcap(col("payment_status")))

# -----------------------------
# SHIPPING
# -----------------------------

silver_shipping = shipping \
    .dropDuplicates(["shipping_id"]) \
    .withColumn("shipping_provider", initcap(col("shipping_provider"))) \
    .withColumn("shipping_status", initcap(col("shipping_status")))

print("All Silver transformations completed!")

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS silver")

In [0]:
silver_customers.write.format("delta").mode("overwrite").saveAsTable("silver.silver_customers")

silver_products.write.format("delta").mode("overwrite").saveAsTable("silver.silver_products")

silver_orders.write.format("delta").mode("overwrite").saveAsTable("silver.silver_orders")

silver_order_items.write.format("delta").mode("overwrite").saveAsTable("silver.silver_order_items")

silver_payments.write.format("delta").mode("overwrite").saveAsTable("silver.silver_payments")

silver_shipping.write.format("delta").mode("overwrite").saveAsTable("silver.silver_shipping")

print("All Silver tables created successfully!")